# Combined XGBoost (Complex) with WOE, IV, and RFE

Same pipeline as the baseline combined model, with a deeper XGBoost: 1000 trees, max_depth=7, lr=0.02, L1/L2 regularization, and early stopping on validation.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "combined_full_enriched.csv"
MODEL_DIR = PROJECT_ROOT / "models"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
IV_THRESHOLD = 0.02
RFE_N_FEATURES = 15
NUMERIC_IV_BINS = 5

N_ESTIMATORS = 1000
MAX_DEPTH = 7
LEARNING_RATE = 0.02
EARLY_STOPPING_ROUNDS = 50


## Load data

In [2]:
df = pd.read_csv(DATA_PATH)
df["target"] = (df["error"] == "error").astype(int)

print(f"Rows: {len(df):,}")
print(df["target"].value_counts())
print()
print(df.groupby(["source", "target"]).size())
df.head(3)

Rows: 30,292
target
1    15345
0    14947
Name: count, dtype: int64

source       target
misprompt    0         14696
             1         14696
realmistake  0           251
             1           649
dtype: int64


,question,llm_model,error,source,split,id,primary_category,secondary_category,explanation,gold_answer,...,question_length_words,question_length_chars,question_complexity_score,has_few_shot_examples,prompt_contains_system_instructions,question_category,is_ambiguous,contains_negation,context_token_count,target
0,Generate a math word problem that satisfies th...,meta-llama/Llama-2-70b-chat-hf,error,realmistake,NaN,NaN,NaN,NaN,NaN,NaN,...,274,1758,10.51,False,False,Reasoning,False,True,342,1
1,Generate a math word problem that satisfies th...,meta-llama/Llama-2-70b-chat-hf,error,realmistake,NaN,NaN,NaN,NaN,NaN,NaN,...,264,1690,10.27,False,False,Reasoning,False,True,330,1
2,Generate a math word problem that satisfies th...,meta-llama/Llama-2-70b-chat-hf,no_error,realmistake,NaN,NaN,NaN,NaN,NaN,NaN,...,103,652,10.50,False,False,Reasoning,False,True,128,0


## Train / validation / test split

In [3]:
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["target"],
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=train_val_df["target"],
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(train_df["target"].value_counts(normalize=True).round(3))

Train: 21,204 | Val: 4,544 | Test: 4,544
target
1    0.507
0    0.493
Name: proportion, dtype: float64


## Feature engineering and WOE encoding

In [4]:
CATEGORICAL_COLUMNS = [
    "model_name",
    "positional_encoding_type",
    "attention_type",
    "tokenizer_type",
    "question_category",
]

BOOLEAN_COLUMNS = [
    "is_open_source",
    "multilingual_support",
    "has_few_shot_examples",
    "prompt_contains_system_instructions",
    "is_ambiguous",
    "contains_negation",
]

NUMERIC_COLUMNS = [
    "context_window_tokens",
    "max_output_tokens",
    "vocab_size",
    "knowledge_cutoff_year",
    "temperature",
    "top_p",
    "top_k",
    "repetition_penalty",
    "frequency_penalty",
    "presence_penalty",
    "max_tokens_requested",
    "stop_sequences_count",
    "galileo_qa_no_rag",
    "galileo_qa_with_rag",
    "galileo_longform",
    "crag_hallucination_rate",
    "crag_accuracy",
    "question_length_words",
    "question_length_chars",
    "question_complexity_score",
    "context_token_count",
]


def compute_woe_maps(frame, columns, target="target"):
    maps = {}
    total_events = frame[target].sum()
    total_non_events = len(frame) - total_events
    for column in columns:
        grouped = frame.groupby(column, dropna=False)[target].agg(["sum", "count"])
        grouped["non_events"] = grouped["count"] - grouped["sum"]
        woe_map = {}
        for value, row in grouped.iterrows():
            event_rate = (row["sum"] + 0.5) / (total_events + 1.0)
            non_event_rate = (row["non_events"] + 0.5) / (total_non_events + 1.0)
            woe_map[value] = float(np.log(event_rate / non_event_rate))
        maps[column] = woe_map
    return maps


def apply_woe(frame, columns, maps):
    transformed = frame.copy()
    for column in columns:
        transformed[f"{column}_woe"] = transformed[column].map(maps[column]).fillna(0.0)
    return transformed


def build_feature_matrix(frame):
    numeric = frame[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
    boolean = frame[BOOLEAN_COLUMNS].astype(int)
    woe_cols = [f"{col}_woe" for col in CATEGORICAL_COLUMNS]
    return pd.concat([numeric, boolean, frame[woe_cols]], axis=1)


woe_maps = compute_woe_maps(train_df, CATEGORICAL_COLUMNS)
train_df = apply_woe(train_df, CATEGORICAL_COLUMNS, woe_maps)
val_df = apply_woe(val_df, CATEGORICAL_COLUMNS, woe_maps)
test_df = apply_woe(test_df, CATEGORICAL_COLUMNS, woe_maps)

feature_names = NUMERIC_COLUMNS + BOOLEAN_COLUMNS + [f"{col}_woe" for col in CATEGORICAL_COLUMNS]

x_train = build_feature_matrix(train_df)
x_val = build_feature_matrix(val_df)
x_test = build_feature_matrix(test_df)

medians = x_train.median(numeric_only=True)
x_train = x_train.fillna(medians)
x_val = x_val.fillna(medians)
x_test = x_test.fillna(medians)

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

print(f"Feature count: {len(feature_names)}")
x_train.head()

Feature count: 32


,context_window_tokens,max_output_tokens,vocab_size,knowledge_cutoff_year,temperature,top_p,top_k,repetition_penalty,frequency_penalty,presence_penalty,...,multilingual_support,has_few_shot_examples,prompt_contains_system_instructions,is_ambiguous,contains_negation,model_name_woe,positional_encoding_type_woe,attention_type_woe,tokenizer_type_woe,question_category_woe
18629,128000,4096,100277,2023,1.0,1.0,50.0,1.2,0.0,0.0,...,1,0,0,0,0,-0.026706,-0.019702,-0.019702,-0.019702,-0.011163
10309,128000,4096,100277,2023,1.0,1.0,50.0,1.2,0.0,0.0,...,1,0,0,0,0,-0.026706,-0.019702,-0.019702,-0.019702,-0.011163
28800,128000,4096,100277,2023,1.0,1.0,50.0,1.2,0.0,0.0,...,1,0,0,0,0,-0.026706,-0.019702,-0.019702,-0.019702,-0.011163
15841,128000,4096,100277,2023,1.0,1.0,50.0,1.2,0.0,0.0,...,1,0,0,0,0,-0.026706,-0.019702,-0.019702,-0.019702,-0.011163
24462,128000,4096,100277,2023,1.0,1.0,50.0,1.2,0.0,0.0,...,1,0,0,0,0,-0.026706,-0.019702,-0.019702,-0.019702,-0.011163


## Information Value (IV) filtering

In [5]:
def compute_iv_grouped(frame, group_col, target="target", observed=False):
    events = frame[target].sum()
    non_events = len(frame) - events
    if events == 0 or non_events == 0:
        return 0.0
    grouped = frame.groupby(group_col, dropna=False, observed=observed)[target].agg(["sum", "count"])
    iv = 0.0
    for _, row in grouped.iterrows():
        bad_dist = row["sum"] / events
        good_dist = (row["count"] - row["sum"]) / non_events
        if bad_dist <= 0 or good_dist <= 0:
            continue
        woe = np.log(bad_dist / good_dist)
        iv += (bad_dist - good_dist) * woe
    return float(iv)


def compute_feature_iv(train_frame, feature_name, medians):
    if feature_name.endswith("_woe"):
        raw_col = feature_name.removesuffix("_woe")
        return compute_iv_grouped(train_frame, raw_col)
    if feature_name in BOOLEAN_COLUMNS:
        return compute_iv_grouped(train_frame, feature_name)
    filled = train_frame[feature_name].fillna(medians.get(feature_name, train_frame[feature_name].median()))
    try:
        binned = pd.qcut(filled, q=NUMERIC_IV_BINS, duplicates="drop")
    except ValueError:
        return 0.0
    temp = pd.DataFrame({"bin": binned, "target": train_frame["target"]})
    return compute_iv_grouped(temp, "bin", observed=True)


iv_scores = {name: compute_feature_iv(train_df, name, medians) for name in feature_names}
iv_df = pd.Series(iv_scores, name="iv").sort_values(ascending=False).reset_index()
iv_df.columns = ["feature", "iv"]
iv_df["passes_threshold"] = iv_df["iv"] >= IV_THRESHOLD
iv_df

,feature,iv,passes_threshold
0,question_length_chars,4.605348e-01,True
1,question_length_words,4.225356e-01,True
2,contains_negation,2.140859e-01,True
3,context_token_count,2.115916e-01,True
4,model_name_woe,3.114921e-02,True
5,tokenizer_type_woe,2.761290e-02,True
6,attention_type_woe,2.761290e-02,True
7,positional_encoding_type_woe,2.761290e-02,True
8,multilingual_support,2.761290e-02,True
9,is_open_source,2.761290e-02,True


In [6]:
iv_features = iv_df.loc[iv_df["passes_threshold"], "feature"].tolist()
if not iv_features:
    iv_features = [iv_df.iloc[0]["feature"]]
print(f"IV selected {len(iv_features)} / {len(feature_names)} features (threshold={IV_THRESHOLD})")
iv_features

IV selected 11 / 32 features (threshold=0.02)


['question_length_chars',
 'question_length_words',
 'contains_negation',
 'context_token_count',
 'model_name_woe',
 'tokenizer_type_woe',
 'attention_type_woe',
 'positional_encoding_type_woe',
 'multilingual_support',
 'is_open_source',
 'question_complexity_score']

## Recursive Feature Elimination (RFE)

In [7]:
scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
n_select = min(RFE_N_FEATURES, len(iv_features))

rfe_estimator = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
)

selector = RFE(estimator=rfe_estimator, n_features_to_select=n_select, step=1)
selector.fit(x_train[iv_features], y_train)

rfe_ranking = pd.DataFrame({
    "feature": iv_features,
    "ranking": selector.ranking_,
    "selected": selector.support_,
}).sort_values("ranking")

rfe_features = rfe_ranking.loc[rfe_ranking["selected"], "feature"].tolist()
print(f"RFE selected {len(rfe_features)} features")
rfe_ranking

RFE selected 11 features


,feature,ranking,selected
0,question_length_chars,1,True
1,question_length_words,1,True
2,contains_negation,1,True
3,context_token_count,1,True
4,model_name_woe,1,True
5,tokenizer_type_woe,1,True
6,attention_type_woe,1,True
7,positional_encoding_type_woe,1,True
8,multilingual_support,1,True
9,is_open_source,1,True


## Train XGBoost on selected features

In [8]:
x_train_sel = x_train[rfe_features]
x_val_sel = x_val[rfe_features]
x_test_sel = x_test[rfe_features]

model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

model.fit(x_train_sel, y_train, eval_set=[(x_val_sel, y_val)], verbose=False)
print(f"Best iteration: {model.best_iteration}")
model

Best iteration: 657


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

## Evaluation

In [9]:
def evaluate_split(name, x, y, model):
    proba = model.predict_proba(x)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        "split": name,
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
        "confusion_matrix": confusion_matrix(y, preds),
        "report": classification_report(y, preds, zero_division=0),
    }


scores = pd.DataFrame([
    evaluate_split("train", x_train_sel, y_train, model),
    evaluate_split("val", x_val_sel, y_val, model),
    evaluate_split("test", x_test_sel, y_test, model),
])

scores[["split", "accuracy", "precision", "recall", "f1", "roc_auc"]]

,split,accuracy,precision,recall,f1,roc_auc
0,train,0.790040,0.810813,0.763709,0.786557,0.874514
1,val,0.752641,0.767484,0.734144,0.750444,0.833879
2,test,0.754842,0.772727,0.731103,0.751339,0.831967


In [10]:
for _, row in scores.iterrows():
    print(f"{row['split'].upper()} confusion matrix:\n{row['confusion_matrix']}\n")
    print(row["report"])
    print("-" * 60)

TRAIN confusion matrix:
[[8549 1914]
 [2538 8203]]

              precision    recall  f1-score   support

           0       0.77      0.82      0.79     10463
           1       0.81      0.76      0.79     10741

    accuracy                           0.79     21204
   macro avg       0.79      0.79      0.79     21204
weighted avg       0.79      0.79      0.79     21204

------------------------------------------------------------
VAL confusion matrix:
[[1730  512]
 [ 612 1690]]

              precision    recall  f1-score   support

           0       0.74      0.77      0.75      2242
           1       0.77      0.73      0.75      2302

    accuracy                           0.75      4544
   macro avg       0.75      0.75      0.75      4544
weighted avg       0.75      0.75      0.75      4544

------------------------------------------------------------
TEST confusion matrix:
[[1747  495]
 [ 619 1683]]

              precision    recall  f1-score   support

           0    

## Inference on test set

In [11]:
test_proba = model.predict_proba(x_test_sel)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

inference_df = test_df[["source", "llm_model", "error", "question_category"]].copy()
inference_df["predicted_error_probability"] = test_proba
inference_df["predicted_label"] = np.where(test_pred == 1, "error", "no_error")
inference_df["correct"] = inference_df["error"] == inference_df["predicted_label"]

print(f"Test accuracy: {inference_df['correct'].mean():.4f}")
print()
print(inference_df.groupby("source")["correct"].mean().round(4))
inference_df.head(10)

Test accuracy: 0.7548

source
misprompt      0.7553
realmistake    0.7398
Name: correct, dtype: float64


,source,llm_model,error,question_category,predicted_error_probability,predicted_label,correct
10985,misprompt,gpt-4o,error,Reasoning,0.582428,error,True
9455,misprompt,gpt-4o,error,Reasoning,0.249615,no_error,False
21103,misprompt,gpt-4o,no_error,Reasoning,0.212152,no_error,True
3726,misprompt,gpt-4o,error,Reasoning,0.749958,error,True
15797,misprompt,gpt-4o,no_error,Reasoning,0.563210,error,False
13481,misprompt,gpt-4o,no_error,Reasoning,0.179306,no_error,True
27451,misprompt,gpt-4o,error,Reasoning,0.406209,no_error,False
26060,misprompt,gpt-4o,no_error,Reasoning,0.837197,error,False
10626,misprompt,gpt-4o,error,Reasoning,0.831463,error,True
29861,misprompt,gpt-4o,no_error,Reasoning,0.752424,error,False


In [12]:
importance = pd.Series(model.feature_importances_, index=rfe_features).sort_values(ascending=False)
importance

contains_negation               0.295293
question_length_words           0.147217
context_token_count             0.137224
question_length_chars           0.118426
question_complexity_score       0.078484
attention_type_woe              0.067606
tokenizer_type_woe              0.063108
model_name_woe                  0.054833
positional_encoding_type_woe    0.037809
multilingual_support            0.000000
is_open_source                  0.000000
dtype: float32

In [13]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_model(MODEL_DIR / "xgboost_combined_woe_iv_rfe_complex.json")

artifact = {
    "feature_names": rfe_features,
    "iv_threshold": IV_THRESHOLD,
    "iv_scores": iv_scores,
    "iv_selected_features": iv_features,
    "rfe_ranking": {row["feature"]: int(row["ranking"]) for _, row in rfe_ranking.iterrows()},
    "woe_maps": {k: {str(key): val for key, val in v.items()} for k, v in woe_maps.items()},
    "numeric_medians": medians.to_dict(),
    "model_params": {
        "n_estimators": N_ESTIMATORS,
        "max_depth": MAX_DEPTH,
        "learning_rate": LEARNING_RATE,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "gamma": 0.1,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "best_iteration": int(model.best_iteration),
    },
}
with (MODEL_DIR / "xgboost_combined_woe_iv_rfe_complex_preprocessing.json").open("w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

metrics_payload = {
    "dataset": "combined_full_enriched.csv",
    "model_variant": "complex",
    "feature_selection": {
        "total_features": len(feature_names),
        "iv_selected": len(iv_features),
        "rfe_selected": len(rfe_features),
        "selected_features": rfe_features,
    },
    "model_params": artifact["model_params"],
}
for split, row in zip(["train", "val", "test"], scores.to_dict("records")):
    metrics_payload[split] = {k: v for k, v in row.items() if k not in ("report", "confusion_matrix")}
    metrics_payload[split]["confusion_matrix"] = row["confusion_matrix"].tolist()
metrics_payload["inference_sample"] = {
    "n_rows": int(len(x_test)),
    "positive_predictions": int(test_pred.sum()),
    "mean_predicted_probability": float(test_proba.mean()),
}
with (MODEL_DIR / "xgboost_combined_woe_iv_rfe_complex_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Saved model to {MODEL_DIR / 'xgboost_combined_woe_iv_rfe_complex.json'}")

Saved model to /Users/konstantine25b/Desktop/Gaia Student Club/Retrival Failure/models/xgboost_combined_woe_iv_rfe_complex.json
